# 🧬 Stage 3: Conditional Molecule GenerationGenerates drug-like molecules conditioned on target protein embeddings.**Prerequisites:** Stage 2 (fine-tuning) must be complete.**Multi-GPU:** Generation runs on GPU 0 (model is loaded per-protein). Multiple datasets can be parallelized across GPUs.

In [ ]:
import os, sysCTDDG_ROOT = os.environ.get("CTDDG_ROOT", os.path.dirname(os.getcwd()))os.environ["CTDDG_ROOT"] = CTDDG_ROOTos.environ["MXNET_CUDNN_LIB_CHECKING"] = "0"os.chdir(CTDDG_ROOT)print(f"Working directory: {os.getcwd()}")

## Configure Generation

In [ ]:
# ── CONFIGURATION ──DATASET_INDEX = 2     # Which dataset's test proteins to generate forRUN = 1               # Run number (for multiple runs)N_SAMPLES = 1000      # Number of molecules to generate per proteinMODEL_NAME = "CTDGD"  # Model name (matches fine-tuning output dir name)print(f"Dataset: {DATASET_INDEX}, Run: {RUN}, Samples: {N_SAMPLES}")

## Run Generation

In [ ]:
%%timeimport subprocessresult = subprocess.run(    [sys.executable, "-m", "jupyter", "nbconvert",     "--to", "notebook", "--execute",     "--ExecutePreprocessor.timeout=86400",     "--ExecutePreprocessor.kernel_name=ctddg_env",     f"--output=Generating_samples_executed.ipynb",     os.path.join(CTDDG_ROOT, "code", "Generating_samples.ipynb")],    capture_output=True, text=True, cwd=CTDDG_ROOT)if result.returncode != 0:    print("STDERR:", result.stderr[-3000:])    print(f"\n❌ Generation failed")else:    print("✅ Generation completed!")

## Quick Preview of Generated Molecules

In [ ]:
import pandas as pdfrom rdkit import Chemfrom rdkit.Chem import Drawfrom IPython.display import displayout_dir = os.path.join(CTDDG_ROOT, "outputs", MODEL_NAME,                       f"Dataset{DATASET_INDEX}", "generated_samples",                       str(N_SAMPLES), f"run{RUN}")csvs = sorted([f for f in os.listdir(out_dir) if f.endswith('.csv')]) if os.path.isdir(out_dir) else []print(f"Found {len(csvs)} protein output files")if csvs:    df = pd.read_csv(os.path.join(out_dir, csvs[0]))    print(f"\nProtein 1: {len(df)} molecules generated")    mols = [Chem.MolFromSmiles(s) for s in df['smiles'].head(12) if Chem.MolFromSmiles(s)]    if mols:        display(Draw.MolsToGridImage(mols, molsPerRow=4))